<a href="https://colab.research.google.com/github/maya-hoff/CS143-Portfolio/blob/main/Copy_of_F7_3_ThinkingQuantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS 195: Natural Language Processing
## Thinking Models and Quantization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F7_3_ThinkingQuantization.ipynb)



## Reminder: Last Demo Day is Wednesday!

* Check out other links on Blackboard
* Portfolios due next week Tuesday

## Today

For our last day in class, we will look at two current topics that connect many things we have done this semester:

- **reasoning / thinking models**: models that generate extra tokens on intermediate reasoning-like behavior before answering
- **quantization**: ways to run models with lower-precision weights so they use less memory

Both deal with engineering trade-offs where you're considering LLM accuracy/performance/quality vs. resource usage


## References

* [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models by Wei et al.](https://arxiv.org/pdf/2201.11903)
* [SLP: Post-training: Instruction Tuning, Alignment, and Test-Time Compute, Chapter 9 of Speech and Language Processing by Daniel Jurafsky & James H. Martin](https://web.stanford.edu/~jurafsky/slp3/9.pdf)
* [HuggingFace Guide on Quantization](https://huggingface.co/docs/optimum/concept_guides/quantization)


In [ ]:
import sys
!{sys.executable} -m pip install -U transformers==4.51.3 accelerate torch torchvision bitsandbytes


## Chain of Thought Prompting

<div>
    <center>
        <img src="https://github.com/ericmanley/s26-CS195NLP/blob/main/images/chain_of_thought_prompting.png?raw=1" width="700">
    </center>
</div>

image source: https://arxiv.org/pdf/2201.11903

## Thinking Out Loud vs Thinking Internally

People often say “chain of thought” when they mean visible step-by-step reasoning. But for a language model, visible reasoning is still generated text.

Important distinctions:

- **Answer**: the final response to the user
- **Explanation**: a short justification intended for the user
- **Reasoning trace**: intermediate reasoning-like text, often longer and more detailed
- **Hidden/internal computation**: what happens inside the model's activations, which we do not directly read from normal generation

For most applications, the final answer matters more than whether the explanation sounds smart.


## One Generation vs. a Reasoning System

When people talk about “thinking models,” they may be describing different things.

In the simplest case, the model makes **one inference call** and generates a longer response. The intermediate reasoning-like text and the final answer are all part of the same response.

But some systems use the model in a loop:

1. generate an initial solution or plan
2. feed that output back into the model with another instruction
3. critique, revise, verify, call a tool, or compare alternatives
4. repeat until the system decides to stop

That second pattern is better described as a **reasoning system** or **agentic workflow**. The model is still doing next-token prediction, but the surrounding program changes what gets fed back into the model.


## Load a Small Chat Model

We will use the same small Qwen model from recent workshops. The goal is not to get state-of-the-art reasoning. The goal is to compare prompting styles and think about tradeoffs.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('Using device:', device)


Using device: cuda


In [ ]:
checkpoint = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint)
model.to(device)
model.eval()


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbe

In [ ]:
# this is the same function we've used now in several notebooks
def generate_chat_response(model_to_use, messages, max_new_tokens=180):
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
    ).to(model_to_use.device)

    outputs = model_to_use.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:],skip_special_tokens=True)


## Asking the model to think

Here's two different prompts for the same question. Let's see if the answer makes a difference, and if so, what the *cost* of the solutions are.

In [ ]:
question = 'A store has 12 boxes. Each box has 8 bags. Each bag has 6 marbles. How many marbles are there total?'

short_messages = [
    {'role': 'system', 'content': 'Give only the final answer.'},
    {'role': 'user', 'content': question},
]

reasoning_messages = [
    {'role': 'system', 'content': 'Solve the problem step by step, then give the final answer.'},
    {'role': 'user', 'content': question},
]

short_answer = generate_chat_response(model,short_messages)
reasoning_answer = generate_chat_response(model,reasoning_messages)

print('SHORT ANSWER')
print(short_answer)
print('tokens:', len(tokenizer.encode(short_answer)))

print('\nREASONING-STYLE ANSWER')
print(reasoning_answer)
print('tokens:', len(tokenizer.encode(reasoning_answer)))


SHORT ANSWER
72
tokens: 2

REASONING-STYLE ANSWER
To solve this problem, we can break it down into smaller steps and calculate the total number of marbles.

Step 1: Calculate the number of bags in one box.
Each box contains 8 bags.

Step 2: Calculate the number of bags in all the boxes combined.
There are 12 boxes, so:
\[ 12 \text{ boxes} \times 8 \text{ bags/box} = 96 \text{ bags} \]

Step 3: Calculate the total number of marbles in all the bags.
Each bag contains 6 marbles.

So, for 96 bags:
\[ 96 \text{ bags} \times 6 \text{ marbles/bag} = 576 \text{ marbles} \]

Therefore, there are a total of 576 marbles.
tokens: 177


## Discussion: Did Reasoning Help?

Try these three prompts
- "If all glips are blops and all blops are zargs, are all glips definitely zargs?"
- "What is the room number for the course CS 195 based on the information provided?"
- "A course meets Monday and Wednesday from 12:30 to 1:45. Does it meet on Friday?"

For each task, discuss:

1. Which prompting style gave the best final answer?
2. Which prompting style gave the clearest explanation?
3. Did longer output always mean better reasoning?
4. Did the model ever explain a wrong answer confidently?
5. Which style would you want in a real application?

## Token Budget: Thinking Costs Something

If a model generates more text, it uses more tokens. More tokens usually means:

- more time
- more compute
- more cost in API settings
- more opportunities to drift or introduce mistakes

Thinking models often improve difficult reasoning tasks by spending more tokens. That does not mean every task needs a long trace.


## Group Exercise: Inspect Reasoning Datasets

Many thinking models are trained or fine-tuned using examples that include reasoning traces. In groups, choose one dataset below and inspect it on Hugging Face.

| Dataset | What to look for |
|---|---|
| [open-thoughts/OpenThoughts-114k](https://huggingface.co/datasets/open-thoughts/OpenThoughts-114k) | Math, science, code, and puzzle problems with model-generated reasoning traces |
| [bespokelabs/Bespoke-Stratos-17k](https://huggingface.co/datasets/bespokelabs/Bespoke-Stratos-17k) | Reasoning traces for coding and math tasks |
| [agentlans/train-of-thought](https://huggingface.co/datasets/agentlans/train-of-thought) | Examples that contrast thinking-on and thinking-off formats |
| [open-r1 reasoning datasets collection](https://huggingface.co/collections/open-r1/reasoning-datasets) | A collection of datasets used in Open-R1 style reasoning experiments |

Guiding questions:

1. What are the main fields in the dataset? For example: prompt, problem, reasoning, answer, source, difficulty, or metadata.
2. Does the dataset show the final answer separately from the reasoning trace?
3. Who or what generated the reasoning traces? Humans, a model, another dataset, or a filtering pipeline?
4. What tasks did you notice are included in the dataset: math, coding, science, logic puzzles, general chat, or something else?
5. If a model were fine-tuned (SFT) on this dataset, what behavior would you expect it to learn?


## Quantization: Smaller Numbers, Smaller Models

A transformer model is mostly a giant collection of learned numbers called **weights**.

Those weights are often stored in numeric formats such as 32-bit floating point, 16-bit floating point, 8-bit integers, or 4-bit quantized values.

**Quantization** stores weights using fewer bits. This can dramatically reduce memory use, which makes it possible to run larger models on smaller hardware.

The tradeoff is that lower precision can sometimes change model quality, speed, or compatibility. The effect is not always obvious from one prompt, so we will treat this as something to observe rather than assume.


## How are model weights stored?

Think about a number like -2305.8

In scientific notation, we could write this as $-2.3058 \times 10^3$

and we set aside some bits to represent the sign, the mantissa, and the exponent (except the exponent is a power of 2 instead of 10)

<div>
    <center>
    <img src="https://github.com/ericmanley/s26-CS195NLP/blob/main/images/floating_point_representation.png?raw=1" width="500">
    </center>
</div>

image source: https://asawicki.info/articles/secrets_of_floating_point_numbers_en.php

Doing it with a smaller bit pattern looks like this

<div>
    <center>
    <img src="https://github.com/ericmanley/s26-CS195NLP/blob/main/images/floating-point-structure.png?raw=1" width="500">
    </center>
</div>

image source: https://developer.nvidia.com/blog/floating-point-8-an-introduction-to-efficient-lower-precision-ai-training/


## But how do you represent a weight with only 4 bits?

4 bits gives you only 16 possible numbers that you can represent!

**Thought experiment:** what if we needed to represent today's high temperature (F) in 4 bits? How could we do it?

*Idea 1:* As a signed integer, we can only represent numbers between -8 and 7. Maybe we just do that but we scale it by 10? Or, maybe times 10 + some offset 40 (since we don't usually need to represent -80F). Then we could represent -40, -30, -20, -10, 0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 110, 120

*Idea 2:* Maybe we represent the numbers with respect to the average high temperature in May in Des Moines. If the average is 72, then we could be *more precise* by representing it as 72 + (bit representation) * 4. Then we can represent 40, 44, 48, 52, 56, 60, 64, 68, 72, 76, 80, 84, 88, 92, 96, 100

*Idea 3:* Make it non-linear, and go with more density around the mean

<div>
    <center>
    <img src="https://github.com/ericmanley/s26-CS195NLP/blob/main/images/temp_as_4_bits.png?raw=1" width="500">
    </center>
</div>

Then, we can pick a different scheme for every month and location!
* We can do the same thing for the collection of weights that happens to be in any given layer/group in the neural network
* The same tricks work with 8 bits but aren't needed as much

## Quantization Experiment

Let's try loading the same model three ways:

- a 16-bit version
- an 8-bit quantized version
- a 4-bit quantized version

Then we will ask each version the same question and compare:

- memory footprint
- response quality
- response style
- whether anything fails because of the runtime

This uses Hugging Face's `BitsAndBytesConfig` pattern. Quantized loading works best on a Colab GPU runtime. If one of the versions fails to load, that is useful information too: deployment details matter.

The memory number below is the model's in-memory footprint, not necessarily the size of the original model files in the Hugging Face cache.


In [31]:
import gc
from transformers import BitsAndBytesConfig

# Free the earlier copy before loading several variants.
try:
    del model
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [32]:
models_by_precision = {}

models_by_precision["16-bit"] = AutoModelForCausalLM.from_pretrained(checkpoint, device_map = "auto",
                                                                     torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32)
print("16-bit model size:",models_by_precision["16-bit"].get_memory_footprint()/1e9,"GB")

models_by_precision["8-bit"] = AutoModelForCausalLM.from_pretrained(checkpoint, device_map = "auto",
                                                                    quantization_config = BitsAndBytesConfig(load_in_8bit=True))
print("8-bit model size:",models_by_precision["8-bit"].get_memory_footprint()/1e9,"GB")

models_by_precision["4-bit"] = AutoModelForCausalLM.from_pretrained(checkpoint, device_map = "auto",
                                                                    quantization_config = BitsAndBytesConfig(
                                                                        load_in_4bit=True,
                                                                        bnb_4bit_quant_type='nf4',
                                                                        bnb_4bit_compute_dtype=torch.float16,
                                                                    ))
print("4-bit model size:",models_by_precision["4-bit"].get_memory_footprint()/1e9,"GB")


16-bit model size: 0.988065664 GB


RuntimeError: 
🚨 CUDA SETUP ERROR: Missing dependency: libnvJitLink.so.13 🚨

CUDA 13.x runtime libraries were not found in the LD_LIBRARY_PATH.

To fix this, make sure that:
1. You have installed CUDA 13.x toolkit on your system
2. The CUDA runtime libraries are in your LD_LIBRARY_PATH

You can add them with (and persist the change by adding the line to your .bashrc):
   export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:/path/to/cuda-13.x/                    lib64

Original error: libnvJitLink.so.13: cannot open shared object file: No such file or directory

🔍 Run this command for detailed diagnostics:
python -m bitsandbytes

If you've tried everything and still have issues:
1. Include ALL version info (operating system, bitsandbytes, pytorch, cuda, python)
2. Describe what you've tried in detail
3. Open an issue with this information:
   https://github.com/bitsandbytes-foundation/bitsandbytes/issues

Native code method attempted to call: lib.cint8_vector_quant()

In [ ]:
comparison_prompt = 'Are there any holidays celebrated on May 4th?'

messages = [
    {'role': 'system', 'content': 'Answer clearly and concisely.'},
    {'role': 'user', 'content': comparison_prompt},
]

for name, loaded_model in models_by_precision.items():
    print('=' * 80)
    print(name)
    print(generate_chat_response(loaded_model, messages))



## Try Your Own Prompt

Choose a prompt where you might notice a quality difference. Good options include:

- a multi-step arithmetic or logic problem
- a short explanation of a course concept
- a coding question
- a question where precision matters
- a question where the model might hallucinate

After running it on the loaded variants, discuss:

1. Did the lower-bit models answer differently?
2. Were the differences meaningful or mostly cosmetic?
3. Was memory savings easier to observe than quality loss?
4. What would you care about more for a real application: memory, speed, cost, or answer quality?


In [33]:
student_prompt = 'Why might a model hallucinate when it does not have enough context?'

messages = [
    {'role': 'system', 'content': 'Answer clearly and concisely.'},
    {'role': 'user', 'content': student_prompt},
]

for name, loaded_model in models_by_precision.items():
    print('=' * 80)
    print(name)
    print(generate_chat_response(loaded_model, messages))


16-bit
A model hallucinating without sufficient context can occur due to several reasons:

1. **Insufficient Data**: The model may not have enough training data, which means the algorithm has learned too few examples to accurately predict what will happen next in the sequence.

2. **Overfitting**: The model might become overly specialized to a specific task or dataset, leading to poor generalization to new contexts.

3. **Temporal Delay**: If the time scale of the prediction is too short compared to the temporal structure of the input data, the model might struggle to make accurate predictions.

4. **Lack of Contextual Information**: Without explicit information about the context (such as past events, current conditions, or previous actions), the model struggles to understand the relationships between inputs and outputs.

5. **Model Bias**: The model's parameters might be biased towards certain outcomes or behaviors that do not align with the observed


## QLoRA: Quantization During Training

Quantization is not only an inference trick.

**QLoRA** is a popular fine-tuning strategy where the base model is loaded in 4-bit precision, but we do not update all of those quantized base weights directly. Instead, we train a small set of additional LoRA adapter weights on top of the quantized model.

That makes fine-tuning much more memory-efficient:

- the large base model stays compact
- the trainable adapter is much smaller
- training can fit on hardware that could not update the full model

So quantization can be part of deployment, but it can also be part of the training recipe.

Hugging Face's [bitsandbytes documentation](https://huggingface.co/docs/transformers/quantization/bitsandbytes) notes that 8-bit and 4-bit training is generally for training extra parameters, which is exactly the LoRA/QLoRA idea.


## Applied Exploration

Perform an SFT fine-tuning (like in `F6_4`) on a subset of one of the reasoning dataset. Generate some examples (e.g., pick a few test examples held out from the dataset) to informally measure what effect training on the reasoning dataset had.


In [34]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import Trainer, TrainingArguments

dataset = load_dataset("gsm8k", "main")
train_subset = dataset["train"].select(range(500))
test_subset = dataset["test"].select(range(10))

def format_example(example):
  return {
      "text": f"Question: {example['question']}\nAnswer: {example['answer']}"
      }

train_subset = train_subset.map(format_example)

model_name = "HuggingFaceTB/SmolLM2-135M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

model.config.pad_token_id = tokenizer.pad_token_id

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_train = train_subset.map(tokenize)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    learning_rate=2e-5,
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train
)

trainer.train()

Step,Training Loss
10,3.042800
20,1.829000
30,1.226100
40,0.939200
50,1.097500
60,0.871000
70,0.942300
80,0.781300
90,0.969100
100,1.055500


TrainOutput(global_step=250, training_loss=1.0733145561218262, metrics={'train_runtime': 67.7469, 'train_samples_per_second': 7.38, 'train_steps_per_second': 3.69, 'total_flos': 81564254208000.0, 'train_loss': 1.0733145561218262, 'epoch': 1.0})

In [36]:
for i in range(3):

    question = test_subset[i]["question"]

    prompt = f"Question: {question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=50
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("\n----------------")
    print("QUESTION:")
    print(question)

    print("\nMODEL RESPONSE:")
    print(response)

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.



----------------
QUESTION:
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

MODEL RESPONSE:
Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Answer: She eats 16 x 3 = <<16*3=48>>48 eggs per day.
She makes 48 x 2 = <<48*2=96>>96 dollars per day.



Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.



----------------
QUESTION:
A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?

MODEL RESPONSE:
Question: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
Answer: The blue fiber is 2*2=<<2*2=4>>4 bolts.
The white fiber is 2*2=<<2*2=4>>4 bolts.
The total fiber is 4+4=<<

----------------
QUESTION:
Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?

MODEL RESPONSE:
Question: Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?
Answer: Josh bought a house for $80,000 and then put in $50,000 in repairs.  This increased the value of the house by 150%.
The house was worth $80,0
